# 🟥 Notebook 3: Real Pub/Sub with Redis

In notebook 1 we built a pub/sub bus from scratch in one Python process. Now let's do the same thing across processes (or even machines) using **Redis Pub/Sub**.

Redis is a tiny, fast in-memory data store. It also happens to ship with a pub/sub feature: clients `SUBSCRIBE` to channels, and `PUBLISH` sends a message to everyone subscribed.

## Learning objectives
- Run Redis with Docker Compose.
- Use the `redis-py` client to publish and subscribe.
- Notice what Redis Pub/Sub does **not** give you (durability, replay).

## 🛠️ Setup

Start Redis:

```bash
cd 01-foundations/messaging-basics
docker compose up -d
```

You can poke at Redis from another terminal with `docker exec -it messaging-basics-redis-1 redis-cli` if you like.

Install Python deps:
```bash
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't show up.

In [ ]:
import redis
import threading
import time

# Connect to the Redis we started with docker compose
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("ping:", r.ping())   # should print True

In [ ]:
# Subscriber runs in a background thread. In a real app this would be a
# separate process or service.
received = []

def subscriber():
    pubsub = r.pubsub()
    pubsub.subscribe("orders")
    for message in pubsub.listen():
        if message["type"] != "message":
            continue            # skip subscribe-confirmation messages
        received.append(message["data"])
        if message["data"] == "STOP":
            return

t = threading.Thread(target=subscriber, daemon=True)
t.start()

time.sleep(0.2)  # give the subscriber a moment to attach
for i in range(3):
    r.publish("orders", f"order-{i}")
r.publish("orders", "STOP")
t.join(timeout=2)

print("subscriber received:", received)

## ⚠️ Redis Pub/Sub is **fire-and-forget**

Redis Pub/Sub is at-most-once delivery: if a subscriber is offline when you publish, it **never sees that message** — there is no replay. There is also no acknowledgment.

Try this experiment yourself: publish a message *before* starting a subscriber and confirm the subscriber sees nothing.

In [ ]:
# No subscribers attached right now.
r.publish("orders", "you-will-never-see-me")

received2 = []
def subscriber2():
    pubsub = r.pubsub()
    pubsub.subscribe("orders")
    for message in pubsub.listen():
        if message["type"] != "message":
            continue
        received2.append(message["data"])
        return

t = threading.Thread(target=subscriber2, daemon=True)
t.start()
time.sleep(0.2)
r.publish("orders", "but-you-will-see-this")
t.join(timeout=2)

print("late subscriber received:", received2)

## ✅ Best practice: Redis **Streams** (durable, replayable)

Redis Pub/Sub is great when missing a message is fine. When it isn't, the same Redis server gives you **Streams** — an append-only log with replay and acknowledgments. Two parts to learn:

1. **Replay** — a late reader can ask for everything from the start (`XREAD` from `0-0`).
2. **Consumer groups** — multiple workers split the load like a queue, and unacked messages stay pending until somebody acks them.

Important mental model: a Streams **consumer group** is *queue-like* (work-sharing across consumers in the group), not pub/sub fan-out. If you want fan-out **plus** durability, you create one consumer group per downstream service — each group reads the same stream independently.

In [ ]:
# Clean slate so the demo is reproducible.
try: r.delete("orders-stream")
except Exception: pass

# Producer writes 3 events BEFORE any consumer exists.
for i in range(3):
    r.xadd("orders-stream", {"order": f"order-{i}"})

# A late reader asks for everything from the very beginning of the stream.
history = r.xread({"orders-stream": "0-0"})
for stream_name, entries in history:
    for entry_id, fields in entries:
        print(f"  replayed {entry_id} -> {fields}")

print("\nNotice: unlike Pub/Sub, the late reader saw messages that were published before it connected.")

In [ ]:
# Consumer group = queue-like work-sharing with acks.
# mkstream=True creates the stream if it doesn't exist; id='0' starts from the beginning.
try:
    r.xgroup_create("orders-stream", "workers", id="0", mkstream=True)
except redis.ResponseError as e:
    # group already exists from a previous run — fine.
    if "BUSYGROUP" not in str(e): raise

# Worker A reads one message but does NOT ack it (simulating a crash).
msgs_a = r.xreadgroup("workers", "worker-A", {"orders-stream": ">"}, count=1)
print("worker-A got (and 'crashed' before acking):", msgs_a)

# Worker B reads the next message and acks it normally.
msgs_b = r.xreadgroup("workers", "worker-B", {"orders-stream": ">"}, count=1)
for _, entries in msgs_b:
    for entry_id, fields in entries:
        print(f"worker-B processed {entry_id} -> {fields}")
        r.xack("orders-stream", "workers", entry_id)

# What's still pending (unacked)?
pending = r.xpending("orders-stream", "workers")
print("\npending summary:", pending)
print("⚠️  unacked entries stay pending until the same consumer retries them, or another consumer claims them via XCLAIM/XAUTOCLAIM.")

## ✅ Recap

- Redis **Pub/Sub** is great for low-latency *real-time* fan-out (live dashboards, chat, presence). Fire-and-forget; no replay; no acks.
- Redis **Streams** add durability, replay, and at-least-once delivery via consumer groups + `XACK`. Use this when missing a message is unacceptable.
- Other production-grade options for durable messaging: **Kafka** (giant durable log), **NATS JetStream**, **RabbitMQ** with persistent queues.
- The mental model from notebook 1 still applies — we just moved the bus from Python memory to a network service.